In [8]:
import dashscope
from dashscope import TextEmbedding
import numpy as np
import faiss
import pickle
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
import os
# 1. 加载环境变量
load_dotenv()
dashscope.api_key = os.getenv("DASHSCOPE_API_KEY")
# 2. 配置模型 (env里面同时配置了阿里和openai的apikey 相同)
model = OpenAIChatModel(
    model_name='qwen-plus',
)

In [9]:
def embed(texts):
    if isinstance(texts, str):
        texts = [texts]

    resp = TextEmbedding.call(
        model="text-embedding-v2",
        input=texts
    )

    return [item["embedding"] for item in resp["output"]["embeddings"]]

In [10]:
import os

def load_txt_files(folder_path):
    docs = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
                docs.append(text)

    return docs
docs = load_txt_files(r"E:\learn\projectNewENV")

def split_with_overlap(text, chunk_size=10, overlap=3):
    #chunk_size每10个字符切一次  overlap每一个块重复3个字符 这样避免上下文丢失
    chunks = []
    start = 0
    i = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        print(f"\n=== chunk {i} ===")
        print(f"start={start}, end={end}")
        print(chunk)# 看看切出来的东西
        start += chunk_size - overlap
        i+=1
    return chunks



In [11]:
import chromadb

# 1. 初始化数据库客户端（数据会保存在本地 ./my_vector_db 文件夹下）
client = chromadb.PersistentClient(path="./my_vector_db")

# 2. 创建或获取一个“集合”（类似关系型数据库的表）
# metadata 设置距离计算方式为余弦相似度 (cosine)
collection = client.get_or_create_collection(name="office_rules", metadata={"hnsw:space": "cosine"})

# 1. 加载所有文档的内容
raw_docs = load_txt_files(r"E:\learn\projectNewENV")

# 2. 关键点：创建一个新列表，用来存放所有文件切好后的片段
all_chunks = []

for text in raw_docs:
    # 对每一个文件内容进行切片
    # 建议将 chunk_size 调大（如 300-500），10 太小了
    file_chunks = split_with_overlap(text, chunk_size=300, overlap=50)

    # 将切好的片段“拍平”存入总列表
    all_chunks.extend(file_chunks)

# 3. 现在的 all_chunks 就是你要的“切片后的返回文件”列表
# 对这些切片生成向量
vectors = embed(all_chunks)

# 4. 存入数据库
collection.add(
    embeddings=vectors,
    documents=all_chunks,  # 使用切片后的列表
    ids=[f"id_{i}" for i in range(len(all_chunks))] # ID 数量要和切片数量一致
)


=== chunk 0 ===
start=0, end=300
公司年假政策：
员工工作满 1 年可享受 5 天带薪年假
年假有效期至次年 3 月 31 日
部门经理审批后方可使用
考勤制度：
工作日上班时间：9:00-18:00
迟到超过 30 分钟记为缺勤

=== chunk 0 ===
start=0, end=300
《2026年近未来智慧城市生活指南》
第一章：智慧交通与低空物流的协同

随着城市低空开放协议的签署，2026年的城市交通进入了三维时代。地面交通由全自动驾驶中枢（CACC）接管，私家车已基本退出核心商业区。取而代之的是“随叫随到”的共享座舱。这些座舱利用毫米波雷达和城市边缘计算节点实时通信，实现了零碰撞和零拥堵。而在上方120米至300米的空域，顺丰与美团的货运无人机正在预设的“虚拟航道”中穿梭。市民在阳台安装的标准化降落舱，可以在下单后的15分钟内接收来自城郊仓库的生鲜物资。这种立体交通网络不仅极大地缩短了通勤时间，还将城市的碳排放量降低了40%以上。
第二章：全息居家办公与柔性空间革命


=== chunk 1 ===
start=250, end=550
不仅极大地缩短了通勤时间，还将城市的碳排放量降低了40%以上。
第二章：全息居家办公与柔性空间革命

远程办公不再局限于屏幕。基于第六代移动通信技术的超低延迟特性，全息投影会议室已成为家庭装修的标配。员工可以坐在家里的多功能转角处，通过光场采集设备与远在千里之外的同事“面对面”交流。为了适应这种变化，现代公寓设计引入了“柔性墙体”概念。白天的办公区在晚上可以通过模块化导轨滑动，转换为卧室或健身房。环境控制系统会自动感应住户的体温与专注度，实时调节室内氧浓度和环境光色温。家不再仅仅是一个睡觉的地方，它是一个能够感知人类情绪并主动进化的动态生物。
第三章：预防医学与生物特征数据主权

在智慧城市中

=== chunk 2 ===
start=500, end=800
，它是一个能够感知人类情绪并主动进化的动态生物。
第三章：预防医学与生物特征数据主权

在智慧城市中，每一个智能水龙头和智能马桶都是体检中心。系统每天都会通过非侵入式采样分析住户的生理指标，并在早期阶段发现潜在的病变风险。这种“预防优于治疗”的模式彻底改变了医疗保险的逻辑。然而，随之而来的是严峻的数据主权挑战。市民的

In [12]:
import os

# ... 之前的初始化代码 (client, collection, embed函数等) ...

# 1. 指定文件并读取
file_path = r"E:\learn\projectNewENV\poliet2.txt"
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()  # 读取全部内容

# 2. 调用切片函数
# 注意：这里直接把读取到的字符串传进去
final_chunks = split_with_overlap(raw_text, chunk_size=200, overlap=30)

# --- 验证环节 ---
print(f"【验证】文档总字数: {len(raw_text)}") # 修正这里，去掉 [0]
print(f"【验证】切片后的总块数: {len(final_chunks)}")
if final_chunks:
    print(f"【验证】第一个切片预览: {final_chunks[0][:50]}...") # 只看前50字
    print(f"【验证】最后一个切片预览: {final_chunks[-1][:50]}...")
# --- 验证完毕 ---

# 3. 向量化并存入数据库
# 注意：一定要把切片后的 final_chunks 传给 embed 和 documents
vectors = embed(final_chunks)

collection.add(
    embeddings=vectors,
    documents=final_chunks,
    ids=[f"id_{i}" for i in range(len(final_chunks))]
)

print("\n✅ 数据已成功切片并存入 ChromaDB 数据库！")


=== chunk 0 ===
start=0, end=200
《2026年近未来智慧城市生活指南》
第一章：智慧交通与低空物流的协同

随着城市低空开放协议的签署，2026年的城市交通进入了三维时代。地面交通由全自动驾驶中枢（CACC）接管，私家车已基本退出核心商业区。取而代之的是“随叫随到”的共享座舱。这些座舱利用毫米波雷达和城市边缘计算节点实时通信，实现了零碰撞和零拥堵。而在上方120米至300米的空域，顺丰与美团的货运无人机正在预设的“虚拟航道”中穿梭

=== chunk 1 ===
start=170, end=370
米的空域，顺丰与美团的货运无人机正在预设的“虚拟航道”中穿梭。市民在阳台安装的标准化降落舱，可以在下单后的15分钟内接收来自城郊仓库的生鲜物资。这种立体交通网络不仅极大地缩短了通勤时间，还将城市的碳排放量降低了40%以上。
第二章：全息居家办公与柔性空间革命

远程办公不再局限于屏幕。基于第六代移动通信技术的超低延迟特性，全息投影会议室已成为家庭装修的标配。员工可以坐在家里的多功能转角处，通过光场

=== chunk 2 ===
start=340, end=540
成为家庭装修的标配。员工可以坐在家里的多功能转角处，通过光场采集设备与远在千里之外的同事“面对面”交流。为了适应这种变化，现代公寓设计引入了“柔性墙体”概念。白天的办公区在晚上可以通过模块化导轨滑动，转换为卧室或健身房。环境控制系统会自动感应住户的体温与专注度，实时调节室内氧浓度和环境光色温。家不再仅仅是一个睡觉的地方，它是一个能够感知人类情绪并主动进化的动态生物。
第三章：预防医学与生物特征数据

=== chunk 3 ===
start=510, end=710
类情绪并主动进化的动态生物。
第三章：预防医学与生物特征数据主权

在智慧城市中，每一个智能水龙头和智能马桶都是体检中心。系统每天都会通过非侵入式采样分析住户的生理指标，并在早期阶段发现潜在的病变风险。这种“预防优于治疗”的模式彻底改变了医疗保险的逻辑。然而，随之而来的是严峻的数据主权挑战。市民的基因图谱和每日代谢数据被存储在基于联邦学习的加密链上。根据《2026数据法案》，任何保险机构和政府部门

=== chunk 4 ===
start=680, end=880
的加密链上。根据《2026数据法案》

In [20]:
import chromadb
db_path = r"E:\learn\projectNewENV\my_vector_db"

# 2. 初始化客户端（它会自动去硬盘找那个路径）
client = chromadb.PersistentClient(path=db_path)
# 3. 重新获取 Collection 对象
# 只要 name 跟你 add 数据时起的名字一样，它就会加载旧数据
collection = client.get_or_create_collection(name="office_rules")

# --- 验证环节 ---
data_count = collection.count()
print(f"📦 数据库加载成功！")
print(f"📊 当前集合中共有 {data_count} 条数据块。")

# 4. 直接查看其中的前 2 条数据
if data_count > 0:
    results = collection.get(limit=4)
    print("\n--- 数据预览 ---")
    for i in range(len(results['ids'])):
        print(f"ID: {results['ids'][i]}")
        print(f" {results['documents'][i]}...\n")

📦 数据库加载成功！
📊 当前集合中共有 8 条数据块。

--- 数据预览 ---
ID: id_0
 公司年假政策：
员工工作满 1 年可享受 5 天带薪年假
年假有效期至次年 3 月 31 日
部门经理审批后方可使用
考勤制度：
工作日上班时间：9:00-18:00
迟到超过 30 分钟记为缺勤...

ID: id_1
 米的空域，顺丰与美团的货运无人机正在预设的“虚拟航道”中穿梭。市民在阳台安装的标准化降落舱，可以在下单后的15分钟内接收来自城郊仓库的生鲜物资。这种立体交通网络不仅极大地缩短了通勤时间，还将城市的碳排放量降低了40%以上。
第二章：全息居家办公与柔性空间革命

远程办公不再局限于屏幕。基于第六代移动通信技术的超低延迟特性，全息投影会议室已成为家庭装修的标配。员工可以坐在家里的多功能转角处，通过光场...

ID: id_2
 成为家庭装修的标配。员工可以坐在家里的多功能转角处，通过光场采集设备与远在千里之外的同事“面对面”交流。为了适应这种变化，现代公寓设计引入了“柔性墙体”概念。白天的办公区在晚上可以通过模块化导轨滑动，转换为卧室或健身房。环境控制系统会自动感应住户的体温与专注度，实时调节室内氧浓度和环境光色温。家不再仅仅是一个睡觉的地方，它是一个能够感知人类情绪并主动进化的动态生物。
第三章：预防医学与生物特征数据...

ID: id_3
 类情绪并主动进化的动态生物。
第三章：预防医学与生物特征数据主权

在智慧城市中，每一个智能水龙头和智能马桶都是体检中心。系统每天都会通过非侵入式采样分析住户的生理指标，并在早期阶段发现潜在的病变风险。这种“预防优于治疗”的模式彻底改变了医疗保险的逻辑。然而，随之而来的是严峻的数据主权挑战。市民的基因图谱和每日代谢数据被存储在基于联邦学习的加密链上。根据《2026数据法案》，任何保险机构和政府部门...



In [21]:
# 2. 定义你的问题
user_query = "2026年如何保护个人的生物特征数据？"

# 3. 将问题向量化 (必须使用和你存数据时一模一样的 embed 函数)
query_vector = embed([user_query])
results = collection.query(
    query_embeddings=query_vector,
    n_results=2,  # 返回最相似的前 2 条
    include=["documents", "metadatas", "distances"] # 包含内容、元数据和相似度距离
)
print(f"\n🔍 问题: {user_query}")
print("-" * 30)
for i in range(len(results['documents'][0])):
    doc = results['documents'][0][i]
    dist = results['distances'][0][i]
    print(f"【匹配结果 {i+1}】(相似度分数: {dist:.4f})")
    print(f"内容内容: {doc[:200]}...") # 打印前200字
    print("-" * 30)


🔍 问题: 2026年如何保护个人的生物特征数据？
------------------------------
【匹配结果 1】(相似度分数: 0.4586)
内容内容: 类情绪并主动进化的动态生物。
第三章：预防医学与生物特征数据主权

在智慧城市中，每一个智能水龙头和智能马桶都是体检中心。系统每天都会通过非侵入式采样分析住户的生理指标，并在早期阶段发现潜在的病变风险。这种“预防优于治疗”的模式彻底改变了医疗保险的逻辑。然而，随之而来的是严峻的数据主权挑战。市民的基因图谱和每日代谢数据被存储在基于联邦学习的加密链上。根据《2026数据法案》，任何保险机构和政府部门...
------------------------------
【匹配结果 2】(相似度分数: 0.6108)
内容内容: 成为家庭装修的标配。员工可以坐在家里的多功能转角处，通过光场采集设备与远在千里之外的同事“面对面”交流。为了适应这种变化，现代公寓设计引入了“柔性墙体”概念。白天的办公区在晚上可以通过模块化导轨滑动，转换为卧室或健身房。环境控制系统会自动感应住户的体温与专注度，实时调节室内氧浓度和环境光色温。家不再仅仅是一个睡觉的地方，它是一个能够感知人类情绪并主动进化的动态生物。
第三章：预防医学与生物特征数据...
------------------------------
